# Supplier Queries

In [1]:
import pandas as pd
import duckdb

import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntRangeSlider, Layout

from juptils import show_pretty

In [2]:

supplier_conditions=[('all', "1=1"),('nippon-metal',"supplier = 'nippon-metal'"),  ('us-steel', "supplier = 'us-steel'")]
WIDTH_50 = Layout(width='50%')

\
Load monthly sales rollup data.  


In [3]:
monthly_sales = pd.read_csv("./csv/monthly-sales.csv", parse_dates=['month'])

## Supplier Yearly Sales

In [4]:
def show_query(supplier_cond):

    sql_template = """
        SELECT 
            EXTRACT(year FROM month) AS year, 
            supplier AS supplier,
            format('{:t,}', SUM(month_amt)) AS tot_amt
        FROM monthly_sales
        WHERE _SUPPLIER_SLUG_
        GROUP BY year, supplier
        ORDER BY year, supplier
        """.replace("_SUPPLIER_SLUG_", supplier_cond)
    
    df = duckdb.query(sql_template).df() 
    df.show_pretty(left=['supplier'])


##############################################
dd=Dropdown(options=supplier_conditions,layout=WIDTH_50, description="supplier")   
interact(show_query, supplier_cond=dd);


interactive(children=(Dropdown(description='supplier', layout=Layout(width='50%'), options=(('all', '1=1'), ('…

# Quarterly Sales

In [7]:
# new
######

sql_template = """
    SELECT
      EXTRACT(year FROM month) AS year,
      'Q' || EXTRACT(QUARTER FROM month) AS qtr,
      supplier AS supplier,
      FORMAT('{:t,}', SUM(month_amt)) AS qtr_amt
      
    FROM monthly_sales
    
    WHERE _SUPPLIER_CONDITION_ 
        AND year BETWEEN _YEAR_MIN_ AND _YEAR_MAX_

    GROUP BY year, qtr, supplier

    ORDER BY year, qtr, supplier;
"""

def show_query(supplier_cond, years):

    (year_min,year_max) = years

    sql = (
        sql_template
        .replace("_SUPPLIER_CONDITION_", supplier_cond)
        .replace("_YEAR_MIN_", str(year_min))
        .replace("_YEAR_MAX_", str(year_max))
    )
    
    df = duckdb.query(sql).df() 
    df.show_pretty(left=['supplier'])


dropdown = Dropdown(options=supplier_conditions, description='Supplier',layout=WIDTH_50)
slider = IntRangeSlider(value=[2023,2026],min=2023, max=2026, step=1, description='Years',layout=WIDTH_50)

_ = interact(show_query, supplier_cond=dropdown, years=slider)

interactive(children=(Dropdown(description='Supplier', layout=Layout(width='50%'), options=(('all', '1=1'), ('…